# Exercise 5.2 — Hyperparameters: choosing the lasso penalty

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day5/day5-02-regularisation.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. Every cell is filled in: run each one in turn.*

Some settings of a model are not learnt from the data: we choose them before
fitting. They are called **hyperparameters**. The lasso has one that matters
most, the size of its penalty, $\lambda$. A small penalty keeps almost every
column; a large one sets more and more coefficients to exactly zero.

We choose $\lambda$ by trying several values and keeping the one that predicts
best on rows the model was not fitted on: a **validation set**. The test set
stays untouched until the very end. This exercise has no blanks: every cell is
filled in. Run each one, read what it prints, and make sure you understand each
step of the search.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.metrics import root_mean_squared_error

The data are in [`data/diabetes_markers.csv`](data/diabetes_markers.csv): the
750 people from earlier exercises. The outcome is `glucose`. There are five
real measurements (`pregnant`, `mass`, `pedigree`, `age`, and `diabetes`, 1 for
yes and 0 for no) and 150 extra columns, `marker_001` to `marker_150`. The
markers are random numbers with no link to glucose, added on purpose. A good
model should ignore them.

In [ ]:
markers = pd.read_csv("data/diabetes_markers.csv")
print(markers.shape)

------------------------------------------------------------------------

### Part 1 — Three sets: training, validation and test

So far we split the data in two. To choose a hyperparameter we need three sets:
the model is fitted on the **training** rows, each penalty is compared on the
**validation** rows, and the chosen model is checked once on the **test** rows.
We split twice with the same function: first 20% for the test set, then a
quarter of the rest for the validation set.

In [ ]:
train_val, test = train_test_split(markers, test_size=0.2, random_state=42)
train, validation = train_test_split(train_val, test_size=0.25, random_state=42)

print(len(train), len(validation), len(test))

# GridSearchCV takes training and validation rows together, plus a marker
# for each row: -1 = used to fit, 0 = used to validate
both = pd.concat([train, validation])
fold = [-1] * len(train) + [0] * len(validation)
val_set = PredefinedSplit(fold)

X_both, y_both = both.drop(columns="glucose"), both["glucose"]
X_test, y_test = test.drop(columns="glucose"), test["glucose"]

450 training rows, 150 validation rows and 150 test rows. `val_set` holds one
way of splitting: 450 rows to fit and 150 to assess.

------------------------------------------------------------------------

### Part 2 — The pipeline, with the penalty left open

The pipeline standardises every predictor, because the penalty treats all
coefficients alike and should not depend on their units. In scikit-learn the
lasso penalty is called `alpha`; we leave it out here, because the search will
set it.

In [ ]:
lasso_pipe = Pipeline([("scale", StandardScaler()),
                       ("model", Lasso(max_iter=100000))])
lasso_pipe

`get_params()` lists every setting of a model with its current value. `alpha`
is the one we will tune:

In [ ]:
lasso_pipe["model"].get_params()

> **Where to find a model's hyperparameters**
>
> Every scikit-learn model lists its settings, with what each one does and its
> default, on its page in the
> [API reference](https://scikit-learn.org/stable/api/index.html). For this model
> see [`Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html).
> Inside a pipeline, put the step name and two underscores in front:
> `model__alpha`.

------------------------------------------------------------------------

### Part 3 — The grid

The grid is the list of penalties to try. `np.logspace(-2, 1, 10)` gives 10
values from $10^{-2} = 0.01$ to $10^{1} = 10$, evenly spaced on a log scale,
which suits penalties. The name says which step and which setting the values
are for: `model__alpha` is the `alpha` of the step called `model`.

In [ ]:
penalty_grid = {"model__alpha": np.logspace(-2, 1, 10)}
print(penalty_grid["model__alpha"].round(3))

Small penalties are close together and large ones far apart, so each step
changes the model by a similar amount.

> **Other kinds of grid**
>
> A regular grid is the simplest choice. scikit-learn also has
> `RandomizedSearchCV()`, which tries random values instead of a fixed list and
> helps when there are several hyperparameters at once. See the scikit-learn
> guide to
> [tuning hyperparameters](https://scikit-learn.org/stable/modules/grid_search.html).

------------------------------------------------------------------------

### Part 4 — The search

`GridSearchCV()` does the work: for each penalty in the grid, it fits the
pipeline on the 450 training rows and scores it on the 150 validation rows.
scikit-learn always maximises its score, so the RMSE comes out negative.

In [ ]:
lasso_search = GridSearchCV(lasso_pipe,
                            param_grid=penalty_grid,
                            cv=val_set,
                            scoring="neg_root_mean_squared_error")
lasso_search.fit(X_both, y_both)

results = pd.DataFrame({"penalty": lasso_search.cv_results_["param_model__alpha"],
                        "val_rmse": -lasso_search.cv_results_["mean_test_score"]})
print(results.astype(float).round(3))

One row per penalty, with its validation RMSE.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(results["penalty"].astype(float), results["val_rmse"], marker="o")
ax.set_xscale("log")
ax.set_xlabel("Penalty (alpha, log scale)")
ax.set_ylabel("Validation RMSE")
ax.set_title("Validation RMSE for each penalty")
plt.show()

In [ ]:
print(lasso_search.best_params_)

**Question!** Which penalty predicts best on the validation rows? What happens
to the RMSE with a smaller and with a larger penalty?

The validation RMSE is 32.64 with the smallest penalty, falls to
26.22 at a penalty of 2.154, and rises again to 28.06 at 10.
Too small a penalty lets the model use the random markers; too large a penalty
also removes real measurements.

------------------------------------------------------------------------

### Part 5 — The final model, checked once on the test rows

After the search, `GridSearchCV()` has already refitted the pipeline with the
chosen penalty on the training and validation rows together (600 people):
that is `best_estimator_`. Check it once on the 150 test rows.

In [ ]:
final_model = lasso_search.best_estimator_
print("test RMSE:", round(root_mean_squared_error(y_test, final_model.predict(X_test)), 2))

Which columns did the lasso keep? These are the coefficients that are not zero:

In [ ]:
coefs = pd.Series(final_model["model"].coef_, index=X_both.columns)
print(coefs[coefs != 0].round(2))

For comparison, the same model without a penalty:

In [ ]:
plain = Pipeline([("scale", StandardScaler()), ("model", LinearRegression())])
plain.fit(X_both, y_both)
print("test RMSE:", round(root_mean_squared_error(y_test, plain.predict(X_test)), 2))

**Question!** How many of the 150 markers did the lasso keep? Which model
predicts the test rows better?

The lasso kept 8 markers, with small coefficients, and set the other 142 to
zero. Of the real measurements it kept `mass`, `age` and `diabetes`. Its
test RMSE is 27.19, against 31.24 without a penalty: the model
without a penalty uses every marker, and the noise it learnt from them costs it
on new people.

> **Key points**
>
> - A hyperparameter is chosen before fitting. For the lasso it is the penalty, `alpha`.
> - Training rows fit the model, validation rows choose the penalty, test rows check the final model once.
> - `GridSearchCV()` fits every value in the grid and scores it on the validation rows; `best_params_` is the choice; `best_estimator_` is the refitted model.
> - The lasso set most of the useless markers to zero, and predicted new people better than the model without a penalty.

> **What comes next**
>
> The lasso and ridge mix (`mixture` in R, `l1_ratio` with `ElasticNet` in
> Python) is a second hyperparameter, tuned the same way by adding it to the grid.
> Exercise 5.3 replaces the single validation set with **cross-validation**,
> which repeats the comparison on several splits of the training rows. The
> reading at the end of the day,
> [Lasso and ridge in a picture](day5-05-lasso-ridge-picture.qmd), shows why the
> lasso can set a coefficient to exactly zero.